# Model Evaluation + Drug-specific Attention (Case Study)

This notebook:
- Exports predictions on a fixed test split
- Exports attention (global + drug-grouped)
- Generates evaluation figures
- Runs a drug case study: pick best/worst drug by median sample-wise PCC and compare attention


In [1]:
import os
import sys
import json

ROOT = "/Users/liuxi/Desktop/RFA_GNN"
if not os.path.exists(ROOT):
    ROOT = os.getcwd()
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print("ROOT=", ROOT)
print("SRC =", SRC)


ROOT= /Users/liuxi/Desktop/RFA_GNN
SRC = /Users/liuxi/Desktop/RFA_GNN/src


## Paths


In [2]:
weights_path = os.path.join(ROOT, "saved_models", "gat_executed.weights.h5")
meta_path = os.path.join(ROOT, "saved_models", "gat_executed.meta.json")
test_ids_path = os.path.splitext(meta_path)[0] + ".test_ids.npy"

out_npz = os.path.join(ROOT, "tmp", "gat_executed.eval.npz")
out_fig_dir = os.path.join(ROOT, "tmp", "gat_executed.fig")

print("weights_path:", weights_path)
print("meta_path   :", meta_path)
print("test_ids    :", test_ids_path)
print("out_npz     :", out_npz)
print("out_fig_dir :", out_fig_dir)


weights_path: /Users/liuxi/Desktop/RFA_GNN/saved_models/gat_executed.weights.h5
meta_path   : /Users/liuxi/Desktop/RFA_GNN/saved_models/gat_executed.meta.json
test_ids    : /Users/liuxi/Desktop/RFA_GNN/saved_models/gat_executed.meta.test_ids.npy
out_npz     : /Users/liuxi/Desktop/RFA_GNN/tmp/gat_executed.eval.npz
out_fig_dir : /Users/liuxi/Desktop/RFA_GNN/tmp/gat_executed.fig


## Export predictions (fixed test split) + attention


In [ ]:
from plot_eval_figures import ExportArgs, export_predictions, generate_test_ids_npy

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

meta_test_ids = str(meta.get("test_ids_npy", "")).strip()
if meta_test_ids and os.path.exists(meta_test_ids):
    test_ids_path = meta_test_ids
else:
    test_ids_path = os.path.splitext(meta_path)[0] + ".test_ids.npy"

if not os.path.exists(test_ids_path):
    print("test_ids not found, generating:", test_ids_path)
    generate_test_ids_npy(meta_path=meta_path, out_path=test_ids_path, root=ROOT)
else:
    print("test_ids found:", test_ids_path)

args = ExportArgs(
    root=ROOT,
    weights=weights_path,
    out=out_npz,
    cell_line=meta.get("cell_line", "ALL"),
    use_landmark_genes=bool(meta.get("use_landmark_genes", True)),
    max_samples=0,
    split_mode=meta.get("split_mode", "cold_drug"),
    test_frac=float(meta.get("test_frac", 0.2)),
    sparse_gat=bool(meta.get("sparse_gat", True)),
    use_drug_fp_embedding=bool(meta.get("use_drug_fp_embedding", False)),
    hidden_dim=int(meta.get("hidden_dim", 64)),
    num_heads=int(meta.get("num_heads", 4)),
    dropout=float(meta.get("dropout", 0.2)),
    attention_layers=int(meta.get("attention_layers", 4)),
    per_node_head=bool(meta.get("per_node_head", True)),
    no_cell_embedding=bool(meta.get("no_cell_embedding", False)),
    no_residualize_target_by_cell=bool(meta.get("no_residualize_target_by_cell", False)),
    eval_drug_zero=True,
    eval_drug_shuffle=True,
    eval_sanity_seed=0,
    eval_sanity_max_eval=20000,
    test_ids_npy=test_ids_path,
    export_attention=True,
    attention_max_samples=2000,
    attention_batch_size=64,
    attention_group_by="drug",
    attention_groups="",
    attention_top_k_groups=12,
)

export_predictions(args)
print("Saved:", out_npz)


test_ids found: /Users/liuxi/Desktop/RFA_GNN/saved_models/gat_executed.meta.test_ids.npy
正在加载 RFA 数据 (Landmark Mode: False)...
CSV路径: CTL=/Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_ctl_n188708x12328.h5, TRT=/Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_trt_cp_n1805898x12328.h5
目标基因数: 12328
正在读取元数据: /Users/liuxi/Desktop/RFA_GNN/data/siginfo_beta.txt
过滤后元数据记录数: 117284
发现 57929 个 Control 样本，涉及 162 个细胞系。
正在加载所有 Control 数据以计算均值...
正在加载数据文件: /Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_ctl_n188708x12328.h5 ...
  H5 (h5py): 匹配到 57929 个样本 (axis0/Index). 读取中...
  读取耗时: 23.10s
  Gene ID 匹配: Index=0, Columns=12328
  形状检查: Rows=57929, Target=12328


## Generate figures


In [ ]:
import importlib
import plot_eval_figures as pef
importlib.reload(pef)
from IPython.display import Image, display

os.makedirs(out_fig_dir, exist_ok=True)
run = pef.load_eval_npz(out_npz)
tag = os.path.splitext(os.path.basename(out_npz))[0]
print("tag=", tag)


## UMAP (PCA(40) → UMAP(k=15))


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import umap

y_true_all = np.asarray(run["y_true"], dtype=np.float32)
y_pred_all = np.asarray(run["y_pred"], dtype=np.float32)
resid_all = y_pred_all - y_true_all

cell_all = np.asarray(run.get("cell_names", []), dtype=str)
drug_all = np.asarray(run.get("drug_ids", []), dtype=str)
plate_all = np.asarray(run.get("det_plate_ids", []), dtype=str)
batch_all = np.asarray(run.get("batch_ids", []), dtype=str)

n = int(y_true_all.shape[0])
rng = np.random.default_rng(0)
max_umap_samples = 8000
idx = np.arange(n)
if n > max_umap_samples:
    idx = rng.choice(n, size=max_umap_samples, replace=False)
idx = np.sort(idx)

def _pca_umap(X, random_state=0):
    X = np.asarray(X, dtype=np.float32)
    Xp = PCA(n_components=40, random_state=random_state).fit_transform(X)
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=random_state).fit_transform(Xp)
    return emb

umap_true = _pca_umap(y_true_all[idx], random_state=0)
umap_pred = _pca_umap(y_pred_all[idx], random_state=1)
umap_resid = _pca_umap(resid_all[idx], random_state=2)

cell = cell_all[idx] if cell_all.size else None
drug = drug_all[idx] if drug_all.size else None
plate = plate_all[idx] if plate_all.size else None
batch = batch_all[idx] if batch_all.size else None

print("UMAP samples:", len(idx))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

color_by = "drug"  # cell | drug | plate | batch
top_k = 200

def _topk_map(labels, top_k):
    labels = np.asarray(labels, dtype=str)
    uniq, cnt = np.unique(labels, return_counts=True)
    order = uniq[np.argsort(-cnt)]
    keep = set(order[: int(top_k)].tolist())
    out = np.array([x if x in keep else "Other" for x in labels], dtype=str)
    return out

def _plot(ax, emb, labels, title, legend):
    if labels is None or len(labels) == 0:
        ax.scatter(emb[:, 0], emb[:, 1], s=2, alpha=0.35)
        ax.set_title(title)
        return

    labels2 = labels
    #legend = False
    if color_by in {"cell", "drug"}:
        labels2 = _topk_map(labels, top_k=top_k)
       

    uniq = np.unique(labels2)
    if legend:
        cmap = plt.cm.tab20(np.linspace(0, 1, len(uniq)))
        for i, u in enumerate(uniq):
            m = labels2 == u
            ax.scatter(emb[m, 0], emb[m, 1], s=3, alpha=0.35, color=cmap[i], label=f"{u} (n={int(np.sum(m))})")
        ax.legend(fontsize=7, frameon=False, loc="best")
    else:
        codes = np.unique(labels2, return_inverse=True)[1]
        sc = ax.scatter(emb[:, 0], emb[:, 1], s=2, alpha=0.35, c=codes, cmap="turbo")
        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

labels = None
if color_by == "cell":
    labels = cell
elif color_by == "drug":
    labels = drug
elif color_by == "plate":
    labels = plate
elif color_by == "batch":
    labels = batch
is_legend = False if color_by == "drug" else True
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), dpi=160)
_plot(axes[0], umap_true, labels, "UMAP(y_true)", legend=is_legend)
_plot(axes[1], umap_pred, labels, "UMAP(y_pred)", legend=is_legend)
_plot(axes[2], umap_resid, labels, "UMAP(y_pred - y_true)", legend=is_legend)
plt.suptitle(f"{tag} | PCA(40) → UMAP(k=15) | color_by={color_by}")
plt.tight_layout()

out_path = os.path.join(out_fig_dir, f"{tag}_umap_{color_by}.png")
plt.savefig(out_path)
plt.close()
display(Image(filename=out_path))


In [ ]:
pef.plot_true_pred_scatter(run, out_path=os.path.join(out_fig_dir, f"{tag}_scatter_all.png"))
display(Image(filename=os.path.join(out_fig_dir, f"{tag}_scatter_all.png")))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cell_names = np.asarray(run["cell_names"], dtype=str)
uniq, counts = np.unique(cell_names, return_counts=True)
top_cells = uniq[np.argsort(-counts)][:10]
print("top_cells:", top_cells.tolist())

y_true = np.asarray(run["y_true"], dtype=np.float32)
y_pred = np.asarray(run["y_pred"], dtype=np.float32)

rng = np.random.default_rng(0)
max_points_per_cell = 30000
max_samples_per_cell = 200

plt.figure(figsize=(6.0, 5.5), dpi=150)
colors = plt.cm.tab10(np.linspace(0, 1, len(top_cells)))
for i, c in enumerate(top_cells):
    m = cell_names == str(c)
    idx = np.where(m)[0]
    if len(idx) == 0:
        continue
    if len(idx) > max_samples_per_cell:
        idx = rng.choice(idx, size=max_samples_per_cell, replace=False)
    yt = y_true[idx].reshape(-1)
    yp = y_pred[idx].reshape(-1)
    if len(yt) > max_points_per_cell:
        sel = rng.choice(len(yt), size=max_points_per_cell, replace=False)
        yt = yt[sel]
        yp = yp[sel]
    plt.scatter(yt, yp, s=2, alpha=0.15, color=colors[i], label=f"{c} (n={int(np.sum(m))})")

mn = float(min(np.min(y_true), np.min(y_pred)))
mx = float(max(np.max(y_true), np.max(y_pred)))
plt.plot([mn, mx], [mn, mx], color="black", linewidth=1)
plt.xlabel("True")
plt.ylabel("Pred")
plt.title(f"{tag}: True vs Pred (Top-10 cells, colored)")
plt.legend(fontsize=7, frameon=False, loc="best")
plt.tight_layout()

out_path = os.path.join(out_fig_dir, f"{tag}_scatter_cells_top10.png")
plt.savefig(out_path)
plt.close()
display(Image(filename=out_path))


In [ ]:
pef.plot_sample_pcc_by_cell(run, out_path=os.path.join(out_fig_dir, f"{tag}_pcc_by_cell.png"), top_k=6)
display(Image(filename=os.path.join(out_fig_dir, f"{tag}_pcc_by_cell.png")))


In [ ]:
pef.plot_sample_pcc_by_drug(run, out_path=os.path.join(out_fig_dir, f"{tag}_pcc_by_drug.png"), top_k=12)
display(Image(filename=os.path.join(out_fig_dir, f"{tag}_pcc_by_drug.png")))


In [ ]:
pef.plot_sample_mse_by_cell(run, out_path=os.path.join(out_fig_dir, f"{tag}_mse_by_cell.png"), top_k=6)
display(Image(filename=os.path.join(out_fig_dir, f"{tag}_mse_by_cell.png")))


In [ ]:
pef.plot_sanity_pcc(run, out_path=os.path.join(out_fig_dir, f"{tag}_sanity_pcc.png"))
display(Image(filename=os.path.join(out_fig_dir, f"{tag}_sanity_pcc.png")))


In [ ]:
pef.plot_delta_pcc(run, out_path=os.path.join(out_fig_dir, f"{tag}_delta_pcc.png"))
display(Image(filename=os.path.join(out_fig_dir, f"{tag}_delta_pcc.png")))


In [ ]:
p_attn = os.path.join(out_fig_dir, f"{tag}_attn_top_genes.png")
pef.plot_attention_top_genes(run, out_path=p_attn, top_k=20, layer=-1)
display(Image(filename=p_attn))


In [ ]:
df_edges = pef.export_attention_top_edges_csv(
    run,
    out_path=os.path.join(out_fig_dir, f"{tag}_attn_top_edges.csv"),
    top_k=50,
    layer=-1,
)
df_edges.head(15)


In [ ]:
import pandas as pd

gene_info_path = os.path.join(ROOT, "data", "GSE92742_Broad_LINCS_gene_info.txt")
gi = pd.read_csv(gene_info_path, sep="\t")
entrez2symbol = dict(zip(gi["pr_gene_id"].astype(str), gi["pr_gene_symbol"].astype(str)))

df_edges["src_symbol"] = df_edges["src_gene"].astype(str).map(entrez2symbol)
df_edges["dst_symbol"] = df_edges["dst_gene"].astype(str).map(entrez2symbol)

df_edges[["src_symbol", "dst_symbol", "attention"]].head(15)


## Case study (drug-specific attention)


In [ ]:
import numpy as np
import pandas as pd

att = run.get("attention", {})
groups = att.get("group_attention_edge_mean", {})
if not isinstance(groups, dict) or len(groups) == 0:
    raise ValueError("No group attention found. Re-run export with attention_group_by=\"drug\".")

drug_ids = np.asarray(run["drug_ids"], dtype=str)
pcc = np.asarray(run["sample_pcc"], dtype=float)

rows = []
for d in sorted(groups.keys()):
    m = drug_ids == str(d)
    if int(np.sum(m)) == 0:
        continue
    vals = pcc[m]
    rows.append({"drug": str(d), "n": int(len(vals)), "pcc_median": float(np.median(vals)), "pcc_mean": float(np.mean(vals))})

df_drug = pd.DataFrame(rows).sort_values(["pcc_median", "n"], ascending=[False, False]).reset_index(drop=True)
display(df_drug)

best_drug = df_drug.iloc[0]["drug"]
worst_drug = df_drug.iloc[-1]["drug"]
print("best_drug =", best_drug)
print("worst_drug=", worst_drug)


In [ ]:
p_best = os.path.join(out_fig_dir, f"{tag}_attn_top_genes_best.png")
p_worst = os.path.join(out_fig_dir, f"{tag}_attn_top_genes_worst.png")

pef.plot_attention_top_genes(run, out_path=p_best, top_k=20, layer=-1, group=best_drug)
pef.plot_attention_top_genes(run, out_path=p_worst, top_k=20, layer=-1, group=worst_drug)

display(Image(filename=p_best))
display(Image(filename=p_worst))


In [ ]:
df_best = pef.export_attention_top_edges_csv(run, out_path=os.path.join(out_fig_dir, f"{tag}_attn_top_edges_best.csv"), top_k=30, layer=-1, group=best_drug)
df_worst = pef.export_attention_top_edges_csv(run, out_path=os.path.join(out_fig_dir, f"{tag}_attn_top_edges_worst.csv"), top_k=30, layer=-1, group=worst_drug)

def with_symbol(df):
    df = df.copy()
    df["src_symbol"] = df["src_gene"].astype(str).map(entrez2symbol)
    df["dst_symbol"] = df["dst_gene"].astype(str).map(entrez2symbol)
    return df

print("\n--- best drug:", best_drug, "---")
display(with_symbol(df_best)[["src_symbol", "dst_symbol", "attention"]].head(15))
print("\n--- worst drug:", worst_drug, "---")
display(with_symbol(df_worst)[["src_symbol", "dst_symbol", "attention"]].head(15))


In [ ]:
edge_index = att["edge_index"]
src = edge_index[0].astype(int)
dst = edge_index[1].astype(int)
non_self = src != dst

A_best = np.asarray(att["group_attention_edge_mean"][best_drug], dtype=float)[-1]
A_worst = np.asarray(att["group_attention_edge_mean"][worst_drug], dtype=float)[-1]

n_nodes = int(np.max(edge_index)) + 1
s_best = np.zeros((n_nodes,), dtype=float)
s_worst = np.zeros((n_nodes,), dtype=float)
np.add.at(s_best, dst[non_self], A_best[non_self])
np.add.at(s_worst, dst[non_self], A_worst[non_self])

genes = np.asarray(run.get("target_genes"), dtype=str)
df = pd.DataFrame({"entrez": genes, "best": s_best, "worst": s_worst})
df["diff_best_minus_worst"] = df["best"] - df["worst"]
df = df.sort_values("diff_best_minus_worst", ascending=False).head(25)
df["symbol"] = df["entrez"].astype(str).map(entrez2symbol)
display(df[["symbol", "entrez", "best", "worst", "diff_best_minus_worst"]])
